# CrossClientRank projection diagnostic
Owner locked .25h/seed; same weights/sample/basis/rcond/gates. No SGD or utility. Private inline W&B key persists in private version history.

In [ ]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=_INLINE_WANDB_KEY or UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


In [ ]:
MH=['v5_books.py', 'v3_methods.py', 'v4_mala.py', 'v7_nystrom.py', 'v9_anchor.py', 'v8_run.py', 'teacher_driver.py', 'v9_run.py', 'v5_run.py', 'new_methods.py', 'prepare.py', 'archs.py', 'v6_schema.py', '__init__.py', 'channel.py', 'v3_dense.py', 'v10_run.py', 'v6_run.py', 'dagsnet.py', 'metrics.py', 'v4_schema.py', 'new_fl.py', 'v7_run.py', 'model_packet.py', 'v6_geometry.py', 'v3_run.py', 'v8_hybrid.py', 'mobility.py', 'p2.py', 'pilot.py', 'v7_basis.py', 'v2_run.py', 'v3_math.py', 'v5_anchor.py', 'v4_run.py', 'v10_energy.py', 'v4_support.py', 'v2_methods.py', 'v2_math.py', 'teacher_base.py']
PROJ=['rank_math.py', 'rank_probe.py']
INPUTS={'train_14_y.npy': '0388e591c9ac97341fd0e72b2069f4efbea5c92b9d8e4ddb92a93ed5f68ae810', 'train_13_y.npy': '185c46c74758f05822c151479f657652dbaad38566d89380d5ef5ecaad5c70d6', 'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'train_14_X.npy': '274b93855a5050c9bf56bb6446fd797e436ef4408425bdb69434f9537e5070da', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'train_0_y.npy': '81d72de22935ae850b5a137cdbeabac90523bcc00a33d585bd379b6fae2e0610', 'train_12_X.npy': 'e297270e4fef9c27f2cecb5b1ad69d458650c5a6f50b9ad65383a5d3ddfe5e35', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'train_0_X.npy': '6eefa0cd099186d7bbe1323261326e3ce98c519411909ca5214373196264459c', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'train_10_y.npy': '0992b94ffd5160830c5ade271c7bf17b4322faac96046f3cd6f134fae3e115d3', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'train_1_y.npy': 'bd68c140cb5e594f23c86c90166cfe080e1a36b45a2ac2c3b27bd4ae678460d2', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'train_7_X.npy': 'f12f40aecf2ad5b28b1bf710986cc517382c106fe7b581d9d5fb47f51b25fa94', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'train_7_y.npy': '1a0cdd31660a9a4365971a4eb28343479004797b44d7c88ceb13289fd9bd377b', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'source_manifest.json': '00fb447676afb708ec0687bec6485348ce3f62543ab282e8ef86d517e9cf5578', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'train_19_y.npy': 'b64e7fe679d9156046e823f135955992f869994fa68fb677a4beefaa680ea0f0', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'train_12_y.npy': 'fcd8f1e70e2c0d0bbf4c5a0ec9addc03f72dd9ea88cbd718b5b2f2547a78c26b', 'train_8_y.npy': '653a8ca50081fa0be465cfb22c48b003a205bcad153b106c551d59233f71745b', 'train_2_X.npy': '7ccbb8b77174417f0849869a5fbe627631c9c46b55d60c1abbf8a71a6d9a4cdc', 'protocol.json': '5af45ca9ce23b09b9331610be46b2162b14fdc04955d883744abf0f64315ea94', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'train_17_y.npy': '61aa1abbd0f517bec9085a17f661ebf8be2dd082376f553a53de442e0c4ca5c6', 'train_16_X.npy': 'e0f07885973ddf63f9ca84e2a03125ef2a155ea58c6ea38c86b4d4c68d8318e9', 'train_11_X.npy': '5cba03a92ec864d1f2141f169e097d26c2d11a434b880b30b3ae9b70c96582cf', 'train_6_X.npy': 'de88ed8dc095576a53b563491216fd70c5ae40aad50be35d58c5b0eef62d0a22', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'train_8_X.npy': '7b4b01e987c056b9156bbaf50589608ed80e7c08c005e0c77344a8ee30a6a5b8', 'train_5_y.npy': '4c908bb0992159f83096d2d3e55fae2bb7d68fa0e3c758215f9327491bb07110', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4', 'warm-proof.json': '7ac401226b124bef5ba7b525e8c9d7a0f9879bd2ba58ab5a8dd6ae83dc715d54', 'train_10_X.npy': '2ab1fb1fe476235a698c36b2550c6c6f724c9412c06f9f3fc171957fece05af8', 'train_15_X.npy': 'e299020f0819218af6e7c2c80f7db42eb2c3961bea5612b14bea7bac3d22d9e9', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'train_17_X.npy': '8e9ce2afadbdd5b68b5f231e4c9ca5264c1121b2246bb089bc3aa8111ebe6ccc', 'train_13_X.npy': 'd3491aaa9e715c886c5104eabaa1a149faa05d401c66a7f013495288e90c3cec', 'train_6_y.npy': '5f0f4a7dfacefeabdb0051e2f822406760b04d0b1187826e8a3364ecd2820978', 'train_9_y.npy': '6d945634c0405cd58df54263fa6cb65293d8dd2565c7279efa2813135b0d7f1a', 'train_18_y.npy': 'c58c0a6ab806325d705aeb079aee445618cd311751ee64e1f63c861fcc7efa4d', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'train_15_y.npy': 'dc08d5dba376554191ff45ea979e860f8414d0f9cf6a0d13f07ead9c5c99c5c8', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'train_16_y.npy': '68c04385b7681bcaba4df93ed962187153d434d55e999e76c72ba485d9e87de0', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'train_11_y.npy': '0bb2c5c82bb953289b45c06356e42b4c55bdbd92202d999d9dab91a65f2c9ce8', 'train_3_y.npy': '0d413dbecc94f09ae50f63f694ac21bcba809345ffd2b9aba49d83813ccaaf9a', 'manifest.json': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'train_4_X.npy': '060a1b6436366464f249867d6979f7cbe26a9c7c5a2ecab6eaa8f4a30d99d221', 'train_5_X.npy': '1ad83b1159a69dbade233ef73ad4418aa792fcfde87c4807726b50bf7092e7e9', 'train_9_X.npy': '13ef6ca16a5a5f522d9f8ae48e4793e8450563a4b5a75cb9d120a87832082ecc', 'train_2_y.npy': 'cdfced8028548985edccbf09b37a266c5df70bad1df8865de79e3727cdbe5b04', 'train_18_X.npy': 'e7a8e89c01201e72be1fb1568b1ef74b41e7ce65f20bd1bb495964748a6b2c5f', 'train_4_y.npy': '9ce240875ae6c999c783e07440a1a9a67f1da67e13f15de7670f9bb4d53d49bc', 'train_19_X.npy': 'b802e20599277d633b2a95853d0a8ad669d901c97ecc5d080487f37eaf1351e9', 'train_1_X.npy': '2b686a2d50645fefc734902cee766a8f19e5b073028cac90752b31be0a9c6e40', 'warm_weights.pt': '6b7bfe22ed9388c5bb66ecfcb0b0dedc8d8b7f385437d3807e785eedbf355c4b', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'train_3_X.npy': '221ea6e95649dbc261d202dfc31b62faf7d1308a90a53b9bbb8ad53380b8e9a6'}
MANIFEST={'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4'}
PROTOCOL_SHA='5af45ca9ce23b09b9331610be46b2162b14fdc04955d883744abf0f64315ea94'

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('protocol.json') if hashlib.sha256(p.read_bytes()).hexdigest()==PROTOCOL_SHA]
assert len(hits)==1,('input missing/ambiguous',hits)
SRC=hits[0];WORK=Path('/kaggle/temp/cross-client-rank');SOURCE=Path('/kaggle/working/source');OUT=Path('/kaggle/working/runs')
assert json.loads((SRC/'input_manifest.json').read_text())==INPUTS
for folder in (WORK/'mh',WORK/'proj',WORK/'knowledge',SOURCE,OUT):folder.mkdir(parents=True,exist_ok=True)
for name,sha in INPUTS.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,name
for name in MH:shutil.copyfile(SRC/name,WORK/'mh'/name)
for name in PROJ:shutil.copyfile(SRC/name,WORK/'proj'/name)
shutil.copyfile(SRC/'meta.json',WORK/'knowledge/meta.json')
for name in MANIFEST:shutil.copyfile(SRC/name,SOURCE/name)
for name in ('manifest.json','protocol.json','warm-proof.json','input_manifest.json'):
    shutil.copyfile(SRC/name,SOURCE/name)
(SOURCE/'source_manifest.json').write_text(json.dumps(MANIFEST,indent=2))
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'proj'))
from mh.dagsnet import build_model,CFG
torch.set_num_threads(1)
warm=json.loads((SRC/'warm-proof.json').read_text())
weights=torch.load(SRC/'warm_weights.pt',map_location='cpu',weights_only=True)
assert weights['round']==2 and weights['fingerprint']==warm['fingerprint'] and len(weights['private'])==20 and not weights['proxy']
model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True)
for sd in weights['private']:
    model.load_state_dict(sd,strict=True)
    assert all(torch.isfinite(v).all() for v in sd.values() if v.is_floating_point())
del model,weights
print('Exact audited CE-r2 weights/20 models/source/train data PASS',flush=True)

DIAGNOSTIC='"""Reproduce frozen probe projection; emit aggregate fit/test diagnostics, never records/gradients."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\nimport numpy as np\nimport torch\n\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom rank_probe import load_model,predict\nfrom rank_math import C,decode_up,decode_down,gradient,basis,projection\n\n\ndef measure(prob,y,logits,delta,fit):\n    design=basis(prob,y,logits);x=design[fit].reshape(-1,32);target=delta[fit].ravel()\n    scale=np.sqrt((x*x).mean(0));keep=scale>1e-12\n    coeff,_,rank,singular=np.linalg.lstsq(x[:,keep]/scale[keep],target,rcond=1e-10)\n    predicted=np.einsum(\'nkd,d->nk\',design[:,:,keep]/scale[keep],coeff)\n    residual=delta-predicted\n    frozen,_=projection(prob,y,logits,delta,fit)\n    assert np.allclose(residual[~fit],frozen,rtol=1e-10,atol=1e-12)\n    def energy(mask):\n        return dict(rows=int(mask.sum()),delta=float(np.mean(delta[mask]**2)),\n                    predicted=float(np.mean(predicted[mask]**2)),residual=float(np.mean(residual[mask]**2)))\n    classes=[]\n    for c in range(C):\n        if not np.any(y==c):classes.append(None);continue\n        item={name:energy(mask&(y==c)) for name,mask in [(\'fit\',fit),(\'test\',~fit)]}\n        e=np.sort((residual[(~fit)&(y==c)]**2).sum(1))[::-1]\n        item[\'test_residual_top1_fraction\']=float(e[0]/e.sum()) if e.sum()>0 else 0.\n        item[\'test_residual_top5_fraction\']=float(e[:5].sum()/e.sum()) if e.sum()>0 else 0.\n        classes.append(item)\n    original=np.zeros(32);original[keep]=coeff/scale[keep]\n    return dict(rank=int(rank),columns=int(keep.sum()),fit_scale=scale.tolist(),\n        coefficients_normalized=coeff.tolist(),coefficients_original=original.tolist(),\n        singular_values=singular.tolist(),condition=float(singular[0]/singular[-1]) if singular[-1]>0 else None,\n        fit=energy(fit),test=energy(~fit),classes=classes)\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True)\n    p.add_argument(\'--seed\',type=int,required=True);p.add_argument(\'--shard\',type=int,required=True);a=p.parse_args()\n    import hashlib\n    identity=hashlib.sha256((a.input/\'protocol.json\').read_bytes()).digest()[:16]\n    replay=json.loads((a.out/\'replay.json\').read_text());pooled,ng,eligible,total=decode_down(bytes.fromhex(replay[\'cohort\']),identity)\n    assert torch.cuda.device_count()==1\n    torch.set_num_threads(1);torch.backends.cuda.matmul.allow_tf32=False\n    torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    weights=torch.load(a.input/\'warm_weights.pt\',map_location=\'cpu\',weights_only=True)\n    assert weights[\'round\']==2 and len(weights[\'private\'])==20 and not weights[\'proxy\']\n    for k in range(a.shard,20,2):\n        n,s,_=decode_up(bytes.fromhex(replay[\'up\'][k]),identity);own=np.divide(s[:2],n,out=np.zeros((2,C)),where=n>0)\n        support=eligible&(n>=32).all(0);x=np.load(a.input/f\'train_{k}_X.npy\',mmap_mode=\'r\');y=np.load(a.input/f\'train_{k}_y.npy\')\n        indexes=[];fit=[]\n        for c in np.flatnonzero(support):\n            available=np.flatnonzero(y==c);rng=np.random.default_rng(np.random.SeedSequence([1729,k,int(c)]))\n            chosen=rng.choice(available,min(256,len(available)),replace=False);indexes.extend(chosen.tolist())\n            fit.extend([True]*(len(chosen)//2)+[False]*(len(chosen)-len(chosen)//2))\n        assert indexes,\'original probe had no common support\'\n        indexes=np.asarray(indexes);fit=np.asarray(fit,dtype=bool);yy=y[indexes]\n        model=load_model(weights,k,\'cuda:0\');z,prob=predict(model,x[indexes],\'cuda:0\')\n        go=gradient(prob,yy,own,n,int(n[0].sum()),support);delta=gradient(prob,yy,pooled,ng,total,support)-go\n        diag=measure(prob,yy,z.astype(\'float64\'),delta,fit)\n        for c,item in enumerate(diag[\'classes\']):\n            reference=replay[\'clients\'][k][\'classes\'][c]\n            assert (item is None)==(reference is None)\n            if item is not None:\n                selected=(~fit)&(yy==c)\n                for name,value in [(\'own_energy\',float(np.mean(go[selected]**2))),(\'delta_energy\',item[\'test\'][\'delta\']),(\'residual_energy\',item[\'test\'][\'residual\'])]:\n                    assert np.isclose(value,reference[name],rtol=1e-6,atol=1e-18),(\'original energy mismatch\',k,c,name,value,reference[name])\n        diag.update(client=k,seed=a.seed,original_energies_reproduced=True)\n        (a.out/f\'diagnostic_{k:02d}.json\').write_text(json.dumps(diag,indent=2,allow_nan=False)+\'\\n\')\n        print(\'DIAGNOSTIC\',a.seed,k,\'condition\',diag[\'condition\'],\'rank\',diag[\'rank\'],flush=True)\n        del model;torch.cuda.empty_cache()\n\n\nif __name__==\'__main__\':main()\n'
DIAGNOSTIC_SHA='51a45e101f2e614203a826e858cc27d8b6c1758e3d4eeb5c1b68ab7519a7153d'
REPLAY={'parent': 'khanhmay0304/mobihet-ccr-probe-s43-20261015-v1', 'audit_sha': '204152849e06a6c5975eb8520f16a21bfac10b4a92c88d967ee79bfe53cd6432', 'cohort': '4d48435201020200ca020000f8f6ea5a5af45ca9ce23b09b9331610be46b216262271b7de883ea3fb03517aaf095e93f812035fe2349ea3f89aa1b3a0f36e43f56b0d638f57ae43fa239555725b3ef3fad76f3147bb2e93ff1d938b45a8de33f23813c448adaed3f142bdf7fe48aec3fdc377850df4fea3f91c1e664a701e03fc2ef3ba2b0d4ea3fce05d1b3b9fce23f37fd6cce2709ef3f01342a3d8a80eb3fc50dbca394e9a53f820d3a0f19f8673ffd1e4f6f0ac6803f54f251aa70fb903f8023156b98df923f9c9b38e45ec5543f208ce818194d623f72fbf8cc9997943f785572e4df5d643f0440f8dd331b7d3f165bf8ab79bf693f0f745ced3139703f7ed6a138537c6a3fef9dc992a94c953f6078f9c81a0c7b3f7d1dcbddaba06c3fdf58030000000000c43d00000000000004a20000000000007b9e0000000000005cad000000000000ca34020000000000092f00000000000067a900000000000025ac0000000000009dc500000000000038640000000000005f190000000000000e4f000000000000c6b1000000000000b05d0300000000001c64000000000000d2ea0b0000000000ed050f0000000000ada10e000000000036a50e000000000055960e0000000000e70e0d0000000000a8140f00000000004a9a0e00000000008c970e0000000000147e0e000000000079df0e0000000000522a0f0000000000a3f40e0000000000eb910e000000000001e60b000000000095df0e00000000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000ffffc789603e817d813c0dd3293d741e263df1b6353d0700143e5f38453c2591313df670343d0b234f3dec19d23c6dc7d43bedbba53c60573a3de0cc613e39dfd13cb1430f0000000000', 'up': ['...'], 'clients': [{'client': 0, 'classes': [{'rows': 128, 'own_energy': 8.430655522694144e-06, 'delta_energy': 1.5760550040561992e-06, 'residual_energy': 1.128754346088404e-05}, {'rows': 128, 'own_energy': 0.04150310928006294, 'delta_energy': 0.023268199483519568, 'residual_energy': 0.006417279361190502}, {'rows': 128, 'own_energy': 0.006190381935842521, 'delta_energy': 0.004476427107050577, 'residual_energy': 0.0023209934202456308}, {'rows': 35, 'own_energy': 0.5203677325494032, 'delta_energy': 0.4968347440601882, 'residual_energy': 0.14049754209797405}, {'rows': 128, 'own_energy': 0.020739441760830965, 'delta_energy': 0.01576907959240431, 'residual_energy': 0.005374095790922998}, {'rows': 128, 'own_energy': 4.830356609922388e-07, 'delta_energy': 1.0591388449609037e-07, 'residual_energy': 2.3440840024472473e-05}, {'rows': 128, 'own_energy': 0.0017859718788950224, 'delta_energy': 9.019395497082732e-05, 'residual_energy': 3.784121319929381e-05}, {'rows': 128, 'own_energy': 0.0003220601711892013, 'delta_energy': 8.520340344687328e-05, 'residual_energy': 2.3656798289547887e-05}, {'rows': 128, 'own_energy': 9.239783787988146e-05, 'delta_energy': 4.082462889920385e-06, 'residual_energy': 2.014493688887446e-05}, {'rows': 128, 'own_energy': 3.3673797325885336e-05, 'delta_energy': 1.9521369676477755e-06, 'residual_energy': 1.188328960624433e-05}, {'rows': 128, 'own_energy': 0.00034107202763520694, 'delta_energy': 2.9378283095706103e-05, 'residual_energy': 2.447034979124961e-05}, {'rows': 128, 'own_energy': 0.009985943494544694, 'delta_energy': 0.00033195633812291666, 'residual_energy': 0.00012201610040521087}, {'rows': 38, 'own_energy': 0.9974620604275085, 'delta_energy': 0.9035347612961526, 'residual_energy': 0.45528785169142993}, {'rows': 128, 'own_energy': 0.021884146364313725, 'delta_energy': 0.016512139856403547, 'residual_energy': 0.00454241347717169}, {'rows': 128, 'own_energy': 2.389525809160187e-06, 'delta_energy': 1.2768041488285594e-08, 'residual_energy': 1.5467167449135385e-05}, {'rows': 128, 'own_energy': 0.010977500270172705, 'delta_energy': 0.005431437812235386, 'residual_energy': 0.001014147469503453}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1863, 'test_rows': 1865}, 'scan_rows': 78372, 'valid_scan_rows': 78372, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 1, 'classes': [{'rows': 32, 'own_energy': 0.24296653993909417, 'delta_energy': 0.23851602460399401, 'residual_energy': 0.0026575019424281165}, {'rows': 128, 'own_energy': 0.0002816071121542646, 'delta_energy': 0.000237276232983009, 'residual_energy': 3.394717442017298e-05}, {'rows': 128, 'own_energy': 2.325589563383582e-05, 'delta_energy': 7.955131549615936e-05, 'residual_energy': 2.3236539472474057e-05}, {'rows': 49, 'own_energy': 0.0899324804915709, 'delta_energy': 0.0791506629209298, 'residual_energy': 0.021893810367192373}, {'rows': 128, 'own_energy': 5.194150692709062e-05, 'delta_energy': 4.0222275985274234e-05, 'residual_energy': 1.0584936658472052e-05}, {'rows': 128, 'own_energy': 1.2466158619333446e-07, 'delta_energy': 2.338740661265647e-07, 'residual_energy': 1.828999229156974e-05}, {'rows': 128, 'own_energy': 0.00035323852211469064, 'delta_energy': 0.00030320054499057285, 'residual_energy': 0.00021724107956823047}, {'rows': 128, 'own_energy': 0.0003694200662145576, 'delta_energy': 7.890428488612449e-05, 'residual_energy': 1.7739602572691404e-05}, {'rows': 128, 'own_energy': 7.093924637644582e-06, 'delta_energy': 2.2671589274735853e-05, 'residual_energy': 2.020631011865021e-05}, {'rows': 59, 'own_energy': 0.05872967908411139, 'delta_energy': 0.05183139625613659, 'residual_energy': 0.032638877874280686}, {'rows': 128, 'own_energy': 4.153715103867665e-05, 'delta_energy': 0.00015189635483768243, 'residual_energy': 2.8516574911487614e-05}, {'rows': 128, 'own_energy': 0.006081016017370232, 'delta_energy': 0.0025645980316714874, 'residual_energy': 0.0008855727132117791}, {'rows': 85, 'own_energy': 0.049056437228542145, 'delta_energy': 0.030447485125953515, 'residual_energy': 0.005880312416121853}, {'rows': 128, 'own_energy': 0.005529765703137887, 'delta_energy': 0.0026333993396338807, 'residual_energy': 0.0005898526867386526}, {'rows': 92, 'own_energy': 0.03634472618549173, 'delta_energy': 0.03462016564858273, 'residual_energy': 0.00308328881535385}, {'rows': 59, 'own_energy': 0.05595865678543426, 'delta_energy': 0.04367265979428449, 'residual_energy': 0.01065776872725586}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1654, 'test_rows': 1656}, 'scan_rows': 40403, 'valid_scan_rows': 40403, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 2, 'classes': [{'rows': 128, 'own_energy': 2.3114302870733296e-06, 'delta_energy': 5.665703304801805e-07, 'residual_energy': 2.6274000657092097e-05}, {'rows': 128, 'own_energy': 0.0008554976744294452, 'delta_energy': 2.326753192375878e-06, 'residual_energy': 3.2429418244712475e-05}, {'rows': 128, 'own_energy': 0.00583798626711723, 'delta_energy': 0.003572169995761555, 'residual_energy': 0.0010666268543438788}, {'rows': 34, 'own_energy': 0.5699398737365917, 'delta_energy': 0.53839035444072, 'residual_energy': 0.14893297593778473}, {'rows': 72, 'own_energy': 0.10403277118735671, 'delta_energy': 0.09274689113111971, 'residual_energy': 0.026267489642341303}, {'rows': 128, 'own_energy': 6.633926085130128e-07, 'delta_energy': 2.7384819663006227e-08, 'residual_energy': 6.956197127397801e-05}, {'rows': 128, 'own_energy': 0.0031459981286388397, 'delta_energy': 0.0005319254151012239, 'residual_energy': 0.00014980242817260963}, {'rows': 128, 'own_energy': 0.0016159078333913852, 'delta_energy': 6.496076702431388e-05, 'residual_energy': 3.5183560018135264e-05}, {'rows': 128, 'own_energy': 2.884636223675027e-06, 'delta_energy': 4.047148763606773e-06, 'residual_energy': 5.075446699805606e-05}, {'rows': 128, 'own_energy': 0.0005540831501931072, 'delta_energy': 0.0002045898687775851, 'residual_energy': 0.00010431628586925632}, {'rows': 128, 'own_energy': 0.00101858207917486, 'delta_energy': 0.0003607793958775203, 'residual_energy': 0.00021305297495779752}, {'rows': 22, 'own_energy': 1.7006429296898185, 'delta_energy': 1.3442263012572684, 'residual_energy': 0.4949384846418506}, {'rows': 37, 'own_energy': 0.4152458813430854, 'delta_energy': 0.366354442300704, 'residual_energy': 0.13381311045202712}, {'rows': 65, 'own_energy': 0.11011650850191257, 'delta_energy': 0.0995075376713944, 'residual_energy': 0.02127943324095715}, {'rows': 128, 'own_energy': 9.955642843556938e-05, 'delta_energy': 7.132707194070089e-05, 'residual_energy': 4.61471709041942e-05}, {'rows': 128, 'own_energy': 5.6205238055900176e-05, 'delta_energy': 9.15302677587544e-05, 'residual_energy': 7.622774677984775e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1637, 'test_rows': 1638}, 'scan_rows': 61618, 'valid_scan_rows': 61618, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 3, 'classes': [{'rows': 128, 'own_energy': 2.7389809523541467e-05, 'delta_energy': 2.2504033398123165e-06, 'residual_energy': 3.7206116609355944e-06}, {'rows': 128, 'own_energy': 0.0007380745855317967, 'delta_energy': 0.0001307679392386876, 'residual_energy': 4.135008135824252e-05}, {'rows': 128, 'own_energy': 0.000105212550084931, 'delta_energy': 3.067890899281999e-05, 'residual_energy': 1.4153862625441352e-05}, {'rows': 128, 'own_energy': 0.0002121663435902673, 'delta_energy': 8.591888407738344e-05, 'residual_energy': 1.799347632111883e-05}, {'rows': 128, 'own_energy': 1.2973327955912565e-05, 'delta_energy': 0.0001347924482721374, 'residual_energy': 3.1681344981855606e-05}, {'rows': 128, 'own_energy': 2.0265193942155807e-06, 'delta_energy': 8.173151260071802e-09, 'residual_energy': 3.916608790120195e-06}, {'rows': 128, 'own_energy': 4.473236481849432e-05, 'delta_energy': 0.00035028993012718745, 'residual_energy': 0.00011846016802222556}, {'rows': 57, 'own_energy': 0.01960739322734851, 'delta_energy': 0.016460384787495997, 'residual_energy': 0.005084509755331022}, {'rows': 128, 'own_energy': 2.5605899306044444e-05, 'delta_energy': 1.4403130664403926e-05, 'residual_energy': 1.0177974722454698e-05}, {'rows': 60, 'own_energy': 0.06033547696463965, 'delta_energy': 0.051748135901000775, 'residual_energy': 0.02273344095017872}, {'rows': 128, 'own_energy': 0.0005451064850167649, 'delta_energy': 7.229240523033641e-06, 'residual_energy': 5.8804853979468525e-06}, {'rows': 69, 'own_energy': 0.026812908526228713, 'delta_energy': 0.0037502659648913565, 'residual_energy': 0.00177865139770272}, {'rows': 128, 'own_energy': 0.0005018726732541764, 'delta_energy': 6.04623174890558e-05, 'residual_energy': 2.1162496971923895e-05}, {'rows': 128, 'own_energy': 0.003478834448983547, 'delta_energy': 0.0016483409530356934, 'residual_energy': 0.00045093657844879747}, {'rows': 128, 'own_energy': 4.3823141237026554e-05, 'delta_energy': 2.1579101380183665e-05, 'residual_energy': 8.713181265055797e-06}, {'rows': 128, 'own_energy': 0.00094386184858213, 'delta_energy': 2.7968969890828702e-05, 'residual_energy': 9.518624153063539e-06}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1849, 'test_rows': 1850}, 'scan_rows': 33858, 'valid_scan_rows': 33858, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 4, 'classes': [{'rows': 128, 'own_energy': 3.1167741900357136e-05, 'delta_energy': 7.391946240689997e-08, 'residual_energy': 4.416956598435366e-06}, {'rows': 128, 'own_energy': 0.004071149254435467, 'delta_energy': 8.805555389295537e-05, 'residual_energy': 3.108552648106537e-05}, {'rows': 128, 'own_energy': 2.3836557252102488e-05, 'delta_energy': 4.93356020511957e-05, 'residual_energy': 3.0813870367848755e-05}, {'rows': 128, 'own_energy': 0.00018762925350474283, 'delta_energy': 7.516327021748722e-05, 'residual_energy': 1.7671082361819053e-05}, {'rows': 128, 'own_energy': 0.0010791319037238801, 'delta_energy': 0.0002578223575697533, 'residual_energy': 0.00011597569363999222}, {'rows': 128, 'own_energy': 3.3333679397168576e-07, 'delta_energy': 2.4289505933899272e-08, 'residual_energy': 8.163985849994077e-06}, {'rows': 128, 'own_energy': 0.0002681572769492244, 'delta_energy': 0.0006182589774691002, 'residual_energy': 0.00015463804233198413}, {'rows': 96, 'own_energy': 0.009742905454354564, 'delta_energy': 0.007765387675714423, 'residual_energy': 0.0028900591430962986}, {'rows': 128, 'own_energy': 0.0010336416659501762, 'delta_energy': 0.00042940338401446435, 'residual_energy': 0.00017727872112401289}, {'rows': 34, 'own_energy': 0.12790828077228072, 'delta_energy': 0.11993582135969504, 'residual_energy': 0.06738710639484159}, {'rows': 128, 'own_energy': 0.0054260721254368125, 'delta_energy': 0.002032090115307565, 'residual_energy': 0.0009915023669351342}, {'rows': 128, 'own_energy': 0.000210328090599182, 'delta_energy': 0.0032565438567827756, 'residual_energy': 0.0010131103249914918}, {'rows': 128, 'own_energy': 0.000264076146804268, 'delta_energy': 7.131326424461325e-05, 'residual_energy': 3.543826502084614e-05}, {'rows': 128, 'own_energy': 5.0038458307100635e-05, 'delta_energy': 0.00012005779704138495, 'residual_energy': 2.718735101179002e-05}, {'rows': 128, 'own_energy': 4.11269003057606e-06, 'delta_energy': 2.641269322822856e-07, 'residual_energy': 7.903739559149005e-06}, {'rows': 73, 'own_energy': 0.051311400377706436, 'delta_energy': 0.03852042421801419, 'residual_energy': 0.016553332038923126}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1867, 'test_rows': 1867}, 'scan_rows': 44034, 'valid_scan_rows': 44034, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 5, 'classes': [{'rows': 128, 'own_energy': 3.2051779556439936e-06, 'delta_energy': 8.479378451082804e-07, 'residual_energy': 1.811484630665696e-05}, {'rows': 128, 'own_energy': 0.0004537361822393162, 'delta_energy': 0.00014957618449241072, 'residual_energy': 5.932800362806424e-05}, {'rows': 107, 'own_energy': 0.021444809806161664, 'delta_energy': 0.016621428873982064, 'residual_energy': 0.006282650313086074}, {'rows': 128, 'own_energy': 0.011863437404927303, 'delta_energy': 0.006390985712449536, 'residual_energy': 0.0017062381373718855}, {'rows': 128, 'own_energy': 0.00012191515428612232, 'delta_energy': 9.517460277901261e-05, 'residual_energy': 3.19019088849937e-05}, {'rows': 128, 'own_energy': 5.2577011384336364e-08, 'delta_energy': 1.181751139834841e-08, 'residual_energy': 2.9897666918307474e-05}, {'rows': 105, 'own_energy': 0.025085625145221245, 'delta_energy': 0.009313711447665943, 'residual_energy': 0.002584774141113002}, {'rows': 128, 'own_energy': 0.0013921936332305714, 'delta_energy': 0.00013440195373678424, 'residual_energy': 4.4837865481257575e-05}, {'rows': 128, 'own_energy': 0.00022849627334140796, 'delta_energy': 2.7258461125783586e-05, 'residual_energy': 3.088613280285986e-05}, {'rows': 74, 'own_energy': 0.06126391222192661, 'delta_energy': 0.053226987189304284, 'residual_energy': 0.02255863787861657}, {'rows': 128, 'own_energy': 5.452832680649802e-05, 'delta_energy': 1.2310427859022167e-05, 'residual_energy': 1.4098696892469407e-05}, {'rows': 40, 'own_energy': 0.16750433050271463, 'delta_energy': 0.08785676044940537, 'residual_energy': 0.04878682313241371}, {'rows': 128, 'own_energy': 0.0018405071930074992, 'delta_energy': 7.290736637117149e-05, 'residual_energy': 3.028466060070092e-05}, {'rows': 34, 'own_energy': 0.19006347388438105, 'delta_energy': 0.17713485000054854, 'residual_energy': 0.06368437175462457}, {'rows': 128, 'own_energy': 9.587141304967057e-06, 'delta_energy': 1.9285535532516154e-06, 'residual_energy': 1.6192861437223512e-05}, {'rows': 128, 'own_energy': 9.102007130781499e-05, 'delta_energy': 3.713207236734732e-07, 'residual_energy': 1.2167874889263077e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1766, 'test_rows': 1768}, 'scan_rows': 44969, 'valid_scan_rows': 44969, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 6, 'classes': [{'rows': 128, 'own_energy': 2.5530394540081994e-06, 'delta_energy': 1.472798256080776e-07, 'residual_energy': 5.8961308289244434e-05}, {'rows': 128, 'own_energy': 0.0024215778095708585, 'delta_energy': 0.00021441176627738073, 'residual_energy': 0.00014302706562930825}, {'rows': 128, 'own_energy': 0.0024898548216693234, 'delta_energy': 0.0014174512832952014, 'residual_energy': 0.0006696989028145248}, {'rows': 128, 'own_energy': 2.8305778773565855e-05, 'delta_energy': 8.167771156125052e-05, 'residual_energy': 6.266550191912326e-05}, {'rows': 128, 'own_energy': 0.004099206873849039, 'delta_energy': 0.0018642528819718257, 'residual_energy': 0.0006728646126801367}, {'rows': 128, 'own_energy': 8.415327639105254e-06, 'delta_energy': 4.020881592383624e-06, 'residual_energy': 7.880225536840217e-05}, {'rows': 128, 'own_energy': 0.0583355365383379, 'delta_energy': 0.027881455369679273, 'residual_energy': 0.007440367337537829}, {'rows': 128, 'own_energy': 0.0024994721754579017, 'delta_energy': 0.0004956750653480051, 'residual_energy': 0.00016154548387123317}, {'rows': 128, 'own_energy': 6.383644384266315e-06, 'delta_energy': 8.56620744512929e-06, 'residual_energy': 9.806315608563241e-05}, {'rows': 128, 'own_energy': 0.006186929771549966, 'delta_energy': 0.004448931561403994, 'residual_energy': 0.0021498309458997016}, {'rows': 106, 'own_energy': 0.137886589072583, 'delta_energy': 0.10986986958355392, 'residual_energy': 0.03180934692622454}, {'rows': 95, 'own_energy': 0.13402736418390432, 'delta_energy': 0.052578480032216164, 'residual_energy': 0.024423891427494314}, {'rows': 74, 'own_energy': 0.2734873146382986, 'delta_energy': 0.22416926061457998, 'residual_energy': 0.0749947424772265}, {'rows': 128, 'own_energy': 0.05160172442424475, 'delta_energy': 0.04229291358312763, 'residual_energy': 0.012031085155972693}, {'rows': 128, 'own_energy': 3.5485708800023724e-07, 'delta_energy': 1.009266862216938e-08, 'residual_energy': 6.420430559811154e-05}, {'rows': 128, 'own_energy': 0.008776640914363911, 'delta_energy': 0.005502098822769825, 'residual_energy': 0.002284972846173292}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1936, 'test_rows': 1939}, 'scan_rows': 78544, 'valid_scan_rows': 78544, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 7, 'classes': [{'rows': 128, 'own_energy': 4.8958719599407596e-05, 'delta_energy': 4.504194336494133e-06, 'residual_energy': 2.1022042528712985e-06}, {'rows': 128, 'own_energy': 0.001061955683486817, 'delta_energy': 1.0390956483860079e-05, 'residual_energy': 4.246486294727514e-06}, {'rows': 128, 'own_energy': 0.0003546496112857355, 'delta_energy': 1.2777313458755534e-05, 'residual_energy': 8.001796624268248e-06}, {'rows': 128, 'own_energy': 0.00016181918400821755, 'delta_energy': 8.03549955719106e-05, 'residual_energy': 1.7514957140025568e-05}, {'rows': 128, 'own_energy': 0.0010124853116585937, 'delta_energy': 0.00017763582763416646, 'residual_energy': 7.445757798350034e-05}, {'rows': 128, 'own_energy': 0.00027677295432177243, 'delta_energy': 0.00023005173730578817, 'residual_energy': 0.00017441276113679246}, {'rows': 128, 'own_energy': 6.391267545679391e-05, 'delta_energy': 0.0005030190911168076, 'residual_energy': 0.00043696481433533877}, {'rows': 128, 'own_energy': 3.7597814401117534e-05, 'delta_energy': 0.00017756828106967921, 'residual_energy': 3.816574177572177e-05}, {'rows': 40, 'own_energy': 0.08309348719386578, 'delta_energy': 0.07422096978194422, 'residual_energy': 0.04157411359784289}, {'rows': 128, 'own_energy': 0.00010116142249500318, 'delta_energy': 1.1448328802239804e-06, 'residual_energy': 1.9843019668998585e-06}, {'rows': 33, 'own_energy': 0.14119251459627558, 'delta_energy': 0.12046541938885741, 'residual_energy': 0.03762603191288079}, {'rows': 62, 'own_energy': 0.03726545807066402, 'delta_energy': 0.007404665481962936, 'residual_energy': 0.003182646248594766}, {'rows': 128, 'own_energy': 0.002452770406075564, 'delta_energy': 0.0002565820286702604, 'residual_energy': 0.0001010200682249716}, {'rows': 128, 'own_energy': 0.0010481974019935702, 'delta_energy': 0.0001947315694999046, 'residual_energy': 4.366618625137548e-05}, {'rows': 128, 'own_energy': 3.0428504981253586e-07, 'delta_energy': 2.2999349364129404e-08, 'residual_energy': 2.6519253361658266e-06}, {'rows': 128, 'own_energy': 0.00275865002812491, 'delta_energy': 0.0003940350935634485, 'residual_energy': 8.660680016915858e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1799, 'test_rows': 1799}, 'scan_rows': 34525, 'valid_scan_rows': 34525, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 8, 'classes': [{'rows': 128, 'own_energy': 1.945762798573222e-06, 'delta_energy': 1.4713953195061802e-06, 'residual_energy': 6.850926674360026e-06}, {'rows': 128, 'own_energy': 0.00306537162985138, 'delta_energy': 0.0002607962609067849, 'residual_energy': 0.00011532338574315871}, {'rows': 128, 'own_energy': 0.0006155243128651034, 'delta_energy': 0.0001366909684887011, 'residual_energy': 7.33487267359383e-05}, {'rows': 128, 'own_energy': 0.00011957283334299516, 'delta_energy': 1.004302773329265e-05, 'residual_energy': 7.867111308359117e-06}, {'rows': 128, 'own_energy': 0.006037302908739214, 'delta_energy': 0.0032500986403601858, 'residual_energy': 0.0008804689420886299}, {'rows': 128, 'own_energy': 6.055892830452946e-06, 'delta_energy': 3.3275133111600575e-06, 'residual_energy': 1.3411319316630458e-05}, {'rows': 128, 'own_energy': 0.0017951655662218884, 'delta_energy': 3.951855788623297e-06, 'residual_energy': 8.11300652408791e-06}, {'rows': 128, 'own_energy': 0.0409536539268065, 'delta_energy': 0.030580213196264163, 'residual_energy': 0.006358073580002912}, {'rows': 73, 'own_energy': 0.02809498863961325, 'delta_energy': 0.025195822825379563, 'residual_energy': 0.011345846484136049}, {'rows': 128, 'own_energy': 0.002036406640921006, 'delta_energy': 0.0011506050498915825, 'residual_energy': 0.0004256286221120129}, {'rows': 128, 'own_energy': 5.1731771317027246e-05, 'delta_energy': 3.351728166027549e-05, 'residual_energy': 2.6893892805803203e-05}, {'rows': 68, 'own_energy': 0.2259263058484483, 'delta_energy': 0.10485472991412825, 'residual_energy': 0.03796523980523114}, {'rows': 128, 'own_energy': 0.0003830429580054081, 'delta_energy': 5.30524076664592e-05, 'residual_energy': 2.163019336178775e-05}, {'rows': 128, 'own_energy': 0.0031190371006258807, 'delta_energy': 0.0008894031956084138, 'residual_energy': 0.00019180801721164006}, {'rows': 128, 'own_energy': 3.095849757890738e-05, 'delta_energy': 1.6175827583830086e-05, 'residual_energy': 1.4569021293566993e-05}, {'rows': 128, 'own_energy': 0.00039665109387803646, 'delta_energy': 2.082326011544268e-05, 'residual_energy': 1.9928847260947046e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1932, 'test_rows': 1933}, 'scan_rows': 65629, 'valid_scan_rows': 65629, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 9, 'classes': [{'rows': 128, 'own_energy': 0.00036866214705337893, 'delta_energy': 0.0001933299951690881, 'residual_energy': 1.1885673988516038e-05}, {'rows': 33, 'own_energy': 0.1479249314983817, 'delta_energy': 0.108710612323874, 'residual_energy': 0.0339341497395025}, {'rows': 44, 'own_energy': 0.08260679216362582, 'delta_energy': 0.0707123192383418, 'residual_energy': 0.032663383118176896}, {'rows': 128, 'own_energy': 0.00014393244411302735, 'delta_energy': 0.00016124800682794837, 'residual_energy': 4.202840131350581e-05}, {'rows': 128, 'own_energy': 0.0022227819708572467, 'delta_energy': 0.0005976263053589026, 'residual_energy': 0.00012355480053873437}, {'rows': 128, 'own_energy': 2.3859283541353292e-06, 'delta_energy': 4.729885071316614e-07, 'residual_energy': 1.1103856556042381e-05}, {'rows': 65, 'own_energy': 0.040109338450353124, 'delta_energy': 0.015748260318353323, 'residual_energy': 0.005322580937825603}, {'rows': 128, 'own_energy': 0.004531705626594822, 'delta_energy': 0.0018401934252260579, 'residual_energy': 0.00020232941971240029}, {'rows': 128, 'own_energy': 0.0008061933587044293, 'delta_energy': 0.00040824952978863274, 'residual_energy': 0.00022306635697363659}, {'rows': 128, 'own_energy': 1.5729755459022285e-06, 'delta_energy': 1.4529250337165408e-05, 'residual_energy': 1.1711994276034357e-05}, {'rows': 128, 'own_energy': 0.00028309358338299246, 'delta_energy': 7.537532675964212e-06, 'residual_energy': 9.500921339567197e-06}, {'rows': 41, 'own_energy': 0.035028788438361456, 'delta_energy': 0.011132328910714062, 'residual_energy': 0.004112003558133101}, {'rows': 128, 'own_energy': 0.009555261449235299, 'delta_energy': 0.002410282998727953, 'residual_energy': 0.0005041164760138058}, {'rows': 128, 'own_energy': 4.77182439848036e-06, 'delta_energy': 7.074253171850384e-05, 'residual_energy': 1.593969875744293e-05}, {'rows': 16, 'own_energy': 1.0393189362361295, 'delta_energy': 1.0277059355087488, 'residual_energy': 0.03166278298138753}, {'rows': 128, 'own_energy': 0.0001067977748415052, 'delta_energy': 5.383510464744915e-05, 'residual_energy': 2.0508735338441675e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1605, 'test_rows': 1607}, 'scan_rows': 29080, 'valid_scan_rows': 29080, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 10, 'classes': [{'rows': 128, 'own_energy': 0.004205709610696093, 'delta_energy': 0.003543726975671941, 'residual_energy': 0.0006634503226521855}, {'rows': 128, 'own_energy': 3.308635488391987e-05, 'delta_energy': 0.000220258243333312, 'residual_energy': 0.00013783825543610937}, {'rows': 128, 'own_energy': 0.00024002229012241752, 'delta_energy': 1.0567746873578326e-05, 'residual_energy': 7.054669678228912e-06}, {'rows': 128, 'own_energy': 0.000843104217928017, 'delta_energy': 3.547583194399163e-05, 'residual_energy': 1.0627131927277891e-05}, {'rows': 128, 'own_energy': 8.347122284246163e-05, 'delta_energy': 6.204477092720288e-05, 'residual_energy': 1.3896348497130936e-05}, {'rows': 128, 'own_energy': 1.5181211233474342e-10, 'delta_energy': 8.247114902568607e-12, 'residual_energy': 4.230904575206022e-06}, {'rows': 91, 'own_energy': 0.05482644578458095, 'delta_energy': 0.0312661570307413, 'residual_energy': 0.010374872875181185}, {'rows': 128, 'own_energy': 0.0005487035443959892, 'delta_energy': 3.0400769735348337e-05, 'residual_energy': 3.7551625612710176e-06}, {'rows': 128, 'own_energy': 0.001525357805281134, 'delta_energy': 0.0007942555455359259, 'residual_energy': 0.00040175537211767755}, {'rows': 128, 'own_energy': 0.0005827329838560287, 'delta_energy': 0.00022752068711151988, 'residual_energy': 0.00012895229463756808}, {'rows': 128, 'own_energy': 0.0009421266521236846, 'delta_energy': 0.0001356164563606463, 'residual_energy': 5.7986896583647145e-05}, {'rows': 128, 'own_energy': 0.00471129026718206, 'delta_energy': 0.009668051563791535, 'residual_energy': 0.0021254291792517747}, None, {'rows': 128, 'own_energy': 0.00014507284832021604, 'delta_energy': 8.231752054723457e-05, 'residual_energy': 1.1460112442536993e-05}, {'rows': 128, 'own_energy': 2.343823559276978e-07, 'delta_energy': 1.1729238861115333e-08, 'residual_energy': 2.8438546929855978e-06}, {'rows': 128, 'own_energy': 2.470463101416856e-05, 'delta_energy': 8.766757149579192e-05, 'residual_energy': 9.50631195747212e-06}], 'projection': {'columns': 32, 'retained_columns': 31, 'fit_rows': 1882, 'test_rows': 1883}, 'scan_rows': 62103, 'valid_scan_rows': 62103, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True]}, {'client': 11, 'classes': [{'rows': 128, 'own_energy': 1.0072545384054935e-05, 'delta_energy': 4.8512554932270385e-08, 'residual_energy': 1.6213722841656365e-05}, {'rows': 56, 'own_energy': 0.17314968524099625, 'delta_energy': 0.12017854321956586, 'residual_energy': 0.012504093804595042}, {'rows': 128, 'own_energy': 0.0031261369578827127, 'delta_energy': 0.0015550139284063562, 'residual_energy': 0.0008258047974653122}, {'rows': 25, 'own_energy': 0.5050751085801742, 'delta_energy': 0.47477214803194684, 'residual_energy': 0.16759028937291426}, {'rows': 128, 'own_energy': 4.164464615485785e-05, 'delta_energy': 9.732789462099504e-05, 'residual_energy': 3.759941729971227e-05}, {'rows': 128, 'own_energy': 2.9345374634762095e-07, 'delta_energy': 1.5147448008302002e-07, 'residual_energy': 4.504224147952536e-05}, {'rows': 53, 'own_energy': 0.14603913508182279, 'delta_energy': 0.09224152993945829, 'residual_energy': 0.033917087532099224}, {'rows': 128, 'own_energy': 0.016690543457495592, 'delta_energy': 0.01058091193052433, 'residual_energy': 0.0027173973593421054}, {'rows': 128, 'own_energy': 3.1502034065113095e-06, 'delta_energy': 8.194393668314896e-06, 'residual_energy': 5.0164815809134605e-05}, {'rows': 128, 'own_energy': 1.7722832128038622e-05, 'delta_energy': 7.883212159841303e-06, 'residual_energy': 1.7967861666056316e-05}, {'rows': 52, 'own_energy': 0.19029849366879026, 'delta_energy': 0.15582711527232151, 'residual_energy': 0.04208755569681617}, {'rows': 128, 'own_energy': 0.021654640185694934, 'delta_energy': 0.0006523890392794, 'residual_energy': 0.00020033202408741512}, {'rows': 128, 'own_energy': 0.0030727953150241107, 'delta_energy': 5.614508989447134e-05, 'residual_energy': 2.8079309226230098e-05}, {'rows': 65, 'own_energy': 0.06415295950036916, 'delta_energy': 0.055353102200441776, 'residual_energy': 0.01961665219983277}, {'rows': 128, 'own_energy': 0.00012261698616680727, 'delta_energy': 8.89391064521749e-05, 'residual_energy': 4.529591672500682e-05}, {'rows': 128, 'own_energy': 0.0006548662267603072, 'delta_energy': 1.363166057213548e-05, 'residual_energy': 1.877776764719683e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1657, 'test_rows': 1659}, 'scan_rows': 42919, 'valid_scan_rows': 42919, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 12, 'classes': [{'rows': 128, 'own_energy': 8.84338438713485e-06, 'delta_energy': 6.321919328161679e-07, 'residual_energy': 1.4437898941984679e-05}, {'rows': 39, 'own_energy': 0.05679682897385892, 'delta_energy': 0.04165970969368544, 'residual_energy': 0.010086980544520924}, {'rows': 128, 'own_energy': 3.2072311133474462e-06, 'delta_energy': 4.0475647866808074e-05, 'residual_energy': 2.7903381133821217e-05}, {'rows': 89, 'own_energy': 0.034215228150914226, 'delta_energy': 0.02577124541795863, 'residual_energy': 0.009156081677477628}, {'rows': 45, 'own_energy': 0.03386302314553023, 'delta_energy': 0.029824238132883565, 'residual_energy': 0.012650201067753428}, {'rows': 128, 'own_energy': 0.0005769419756087662, 'delta_energy': 0.0004948764755200196, 'residual_energy': 0.0003452049965362012}, {'rows': 53, 'own_energy': 0.09508211389358835, 'delta_energy': 0.05278285050245977, 'residual_energy': 0.013915517875267704}, None, {'rows': 128, 'own_energy': 0.00016322018463834227, 'delta_energy': 2.443702007294594e-05, 'residual_energy': 2.5973398164734185e-05}, {'rows': 128, 'own_energy': 0.00013492623963674483, 'delta_energy': 1.6231376577967957e-06, 'residual_energy': 6.834015494906693e-06}, {'rows': 32, 'own_energy': 0.2616865241438971, 'delta_energy': 0.22488314094640705, 'residual_energy': 0.06950387271745584}, {'rows': 32, 'own_energy': 0.050507273761584905, 'delta_energy': 0.025801450384265515, 'residual_energy': 0.009281854623250114}, {'rows': 128, 'own_energy': 0.0021703066272217393, 'delta_energy': 9.516085412082874e-05, 'residual_energy': 3.74230138166273e-05}, {'rows': 128, 'own_energy': 7.623476043956784e-05, 'delta_energy': 0.00016554356883022737, 'residual_energy': 3.835070760747343e-05}, None, {'rows': 128, 'own_energy': 1.2344388548320121e-05, 'delta_energy': 4.265550092290913e-05, 'residual_energy': 2.0865771635847898e-05}], 'projection': {'columns': 32, 'retained_columns': 30, 'fit_rows': 1314, 'test_rows': 1314}, 'scan_rows': 34788, 'valid_scan_rows': 34788, 'support': [True, True, True, True, True, True, True, False, True, True, True, True, True, True, False, True]}, {'client': 13, 'classes': [{'rows': 128, 'own_energy': 0.00075222925166654, 'delta_energy': 0.0004533197485666992, 'residual_energy': 8.028265657597614e-05}, {'rows': 128, 'own_energy': 0.015136357530914664, 'delta_energy': 0.0054539729639986175, 'residual_energy': 0.001163023301073028}, {'rows': 86, 'own_energy': 0.025464241574827382, 'delta_energy': 0.02047343140571074, 'residual_energy': 0.009054243264747885}, {'rows': 128, 'own_energy': 0.001768671113769279, 'delta_energy': 0.00020164008405828496, 'residual_energy': 4.776572251229577e-05}, {'rows': 128, 'own_energy': 3.627240433834955e-05, 'delta_energy': 8.781813274165647e-05, 'residual_energy': 2.50707818158141e-05}, {'rows': 128, 'own_energy': 9.126858606297106e-06, 'delta_energy': 4.534576610546135e-06, 'residual_energy': 3.638419287800146e-05}, {'rows': 52, 'own_energy': 0.1125547763487335, 'delta_energy': 0.07085032221656903, 'residual_energy': 0.017496484737282887}, {'rows': 79, 'own_energy': 0.042115560918014286, 'delta_energy': 0.034700458750972794, 'residual_energy': 0.008712374666923635}, {'rows': 128, 'own_energy': 2.9378736945486557e-05, 'delta_energy': 3.727534210033433e-05, 'residual_energy': 3.955930557714221e-05}, {'rows': 128, 'own_energy': 1.0742006912203262e-05, 'delta_energy': 1.728830589655515e-05, 'residual_energy': 3.3359017117921505e-05}, {'rows': 128, 'own_energy': 0.0034459733340612402, 'delta_energy': 0.0010475340816168766, 'residual_energy': 0.0004752357129631195}, {'rows': 36, 'own_energy': 0.13823711739118616, 'delta_energy': 0.07436251966508808, 'residual_energy': 0.01753983065244273}, {'rows': 128, 'own_energy': 0.004242614397611138, 'delta_energy': 0.0005514907351912451, 'residual_energy': 0.00011938652943973913}, {'rows': 128, 'own_energy': 0.00029344910955447957, 'delta_energy': 2.792327419285036e-05, 'residual_energy': 2.0516100199671597e-05}, {'rows': 128, 'own_energy': 1.6493999568342562e-07, 'delta_energy': 3.1138163640037894e-08, 'residual_energy': 2.3319195820698268e-05}, {'rows': 46, 'own_energy': 0.2220737371072687, 'delta_energy': 0.18527709388867403, 'residual_energy': 0.04743720689744325}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1704, 'test_rows': 1707}, 'scan_rows': 41827, 'valid_scan_rows': 41827, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 14, 'classes': [{'rows': 117, 'own_energy': 0.09960504955225881, 'delta_energy': 0.09586149449774381, 'residual_energy': 0.0077982458785750405}, {'rows': 36, 'own_energy': 0.8011874382525619, 'delta_energy': 0.7005861438801468, 'residual_energy': 0.2563066954127322}, {'rows': 128, 'own_energy': 0.0181709071626582, 'delta_energy': 0.01427025344616615, 'residual_energy': 0.004256470737542827}, {'rows': 128, 'own_energy': 0.007087948779864255, 'delta_energy': 0.002879342286516412, 'residual_energy': 0.0004380532996647239}, {'rows': 89, 'own_energy': 0.16772704371412875, 'delta_energy': 0.14834267135551768, 'residual_energy': 0.031835545777964554}, {'rows': 128, 'own_energy': 1.4105590783604197e-16, 'delta_energy': 1.1389700654943832e-17, 'residual_energy': 9.413209464823654e-05}, {'rows': 128, 'own_energy': 0.0007257873542855406, 'delta_energy': 0.00011768535053167212, 'residual_energy': 7.856300339583739e-05}, {'rows': 128, 'own_energy': 3.5203360860324607e-06, 'delta_energy': 2.284119441604239e-05, 'residual_energy': 4.077901953655133e-05}, {'rows': 128, 'own_energy': 0.0027627643328115526, 'delta_energy': 0.0019824291588295666, 'residual_energy': 0.0008975585113992679}, {'rows': 128, 'own_energy': 0.00015613875962608117, 'delta_energy': 1.6431544151918422e-05, 'residual_energy': 5.654646587842428e-05}, {'rows': 58, 'own_energy': 0.36209657651470295, 'delta_energy': 0.31700638047661617, 'residual_energy': 0.07390618161710999}, {'rows': 128, 'own_energy': 0.009168947857502996, 'delta_energy': 5.999644389075214e-05, 'residual_energy': 5.473090336646154e-05}, {'rows': 128, 'own_energy': 0.0191243873063423, 'delta_energy': 0.0119459698687899, 'residual_energy': 0.006974716624742688}, {'rows': 128, 'own_energy': 0.02271840866301953, 'delta_energy': 0.015900596944560728, 'residual_energy': 0.0038178142882971887}, {'rows': 128, 'own_energy': 4.727985543813982e-09, 'delta_energy': 2.5680711897140474e-10, 'residual_energy': 7.378262076905125e-05}, {'rows': 38, 'own_energy': 0.2254430735954365, 'delta_energy': 0.2068173222054332, 'residual_energy': 0.08272877318673218}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1743, 'test_rows': 1746}, 'scan_rows': 71183, 'valid_scan_rows': 71183, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 15, 'classes': [{'rows': 128, 'own_energy': 3.758190337431225e-05, 'delta_energy': 4.52649792397266e-06, 'residual_energy': 8.38583871844779e-06}, {'rows': 128, 'own_energy': 0.0007783063994409031, 'delta_energy': 7.837929509585214e-05, 'residual_energy': 1.6280617527398716e-05}, {'rows': 87, 'own_energy': 0.013808921604862556, 'delta_energy': 0.00986852765907571, 'residual_energy': 0.005347731160134884}, {'rows': 128, 'own_energy': 1.2051421257696012e-05, 'delta_energy': 0.00014486116918135597, 'residual_energy': 4.782162550283516e-05}, {'rows': 128, 'own_energy': 0.00033099449136976967, 'delta_energy': 4.455304838971878e-05, 'residual_energy': 1.6886053499051096e-05}, {'rows': 128, 'own_energy': 0.0003966525896970006, 'delta_energy': 0.0003181588039354509, 'residual_energy': 0.00013663648147722214}, {'rows': 128, 'own_energy': 0.0004002193474548822, 'delta_energy': 0.00038122673779594036, 'residual_energy': 3.620902360287575e-05}, {'rows': 22, 'own_energy': 0.07419842771698085, 'delta_energy': 0.06852096737727657, 'residual_energy': 0.031918339208676164}, {'rows': 128, 'own_energy': 8.919970153389904e-05, 'delta_energy': 5.214355637788767e-07, 'residual_energy': 1.4115570223582761e-05}, {'rows': 128, 'own_energy': 0.001035391023134565, 'delta_energy': 0.00033716174026418404, 'residual_energy': 0.00011752664610305258}, {'rows': 128, 'own_energy': 2.6636508717188233e-06, 'delta_energy': 9.076555228815643e-05, 'residual_energy': 5.742842090227429e-05}, {'rows': 89, 'own_energy': 0.014189492152076186, 'delta_energy': 2.40618135666037e-05, 'residual_energy': 1.4818473755061481e-05}, {'rows': 128, 'own_energy': 0.00015600728275255246, 'delta_energy': 0.000419200179225971, 'residual_energy': 7.529238354697707e-05}, {'rows': 128, 'own_energy': 0.00013969793013436757, 'delta_energy': 0.00013417632699856994, 'residual_energy': 3.429339868569314e-05}, {'rows': 33, 'own_energy': 0.17692746619496624, 'delta_energy': 0.1727283671471521, 'residual_energy': 0.05896162679246465}, {'rows': 44, 'own_energy': 0.09089764863524687, 'delta_energy': 0.06980329392123695, 'residual_energy': 0.02559355010661989}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1680, 'test_rows': 1683}, 'scan_rows': 28322, 'valid_scan_rows': 28322, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 16, 'classes': [{'rows': 128, 'own_energy': 0.0006413746801970936, 'delta_energy': 0.00038480805311176574, 'residual_energy': 7.85074941073723e-05}, {'rows': 128, 'own_energy': 2.248307025770976e-05, 'delta_energy': 0.00041673727205246186, 'residual_energy': 7.054021148441013e-05}, {'rows': 128, 'own_energy': 0.0003232300776945387, 'delta_energy': 7.141045613268037e-06, 'residual_energy': 7.292492301659982e-06}, {'rows': 128, 'own_energy': 7.998659709604312e-05, 'delta_energy': 0.0002661478746300474, 'residual_energy': 4.853313765014888e-05}, {'rows': 128, 'own_energy': 0.00025896019500209745, 'delta_energy': 6.988743481037586e-05, 'residual_energy': 1.6318071027532223e-05}, {'rows': 128, 'own_energy': 1.567765211431565e-05, 'delta_energy': 9.440618948267823e-06, 'residual_energy': 1.0960350359331245e-05}, {'rows': 128, 'own_energy': 0.002276676464891715, 'delta_energy': 3.298886695806705e-05, 'residual_energy': 1.534570977088275e-05}, {'rows': 35, 'own_energy': 0.01191412958384832, 'delta_energy': 0.009966789009158011, 'residual_energy': 0.0019374633944156544}, {'rows': 43, 'own_energy': 0.04196321877942675, 'delta_energy': 0.03399073102766932, 'residual_energy': 0.01254987691801626}, {'rows': 128, 'own_energy': 2.3552616756858056e-05, 'delta_energy': 7.538658636679975e-06, 'residual_energy': 6.960907178546858e-06}, {'rows': 128, 'own_energy': 0.0013380513314309927, 'delta_energy': 5.778873740589132e-07, 'residual_energy': 2.847724696056687e-06}, {'rows': 36, 'own_energy': 0.008368365016366052, 'delta_energy': 0.001680177890306122, 'residual_energy': 0.0007975482595344086}, {'rows': 128, 'own_energy': 3.473882493696842e-08, 'delta_energy': 1.729448319194028e-06, 'residual_energy': 8.568195272580583e-06}, {'rows': 128, 'own_energy': 0.00030390510850901854, 'delta_energy': 7.713350549316575e-05, 'residual_energy': 1.2286401819748286e-05}, {'rows': 33, 'own_energy': 0.09089046567384641, 'delta_energy': 0.0878082402827171, 'residual_energy': 0.025982572638291873}, {'rows': 50, 'own_energy': 0.03215718605397468, 'delta_energy': 0.020761018654537347, 'residual_energy': 0.004444512319056627}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1602, 'test_rows': 1605}, 'scan_rows': 19882, 'valid_scan_rows': 19882, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 17, 'classes': [{'rows': 128, 'own_energy': 0.021055597695715238, 'delta_energy': 0.019462283704563477, 'residual_energy': 0.0025296814547886796}, {'rows': 128, 'own_energy': 0.0024351358690784778, 'delta_energy': 0.0006284239775227014, 'residual_energy': 0.00026910321460653694}, {'rows': 128, 'own_energy': 0.00012118690135628619, 'delta_energy': 2.2350744812923315e-06, 'residual_energy': 3.63253626101153e-05}, {'rows': 99, 'own_energy': 0.38104374395756196, 'delta_energy': 0.35263708716073383, 'residual_energy': 0.12674164019780212}, {'rows': 128, 'own_energy': 0.006937347975163783, 'delta_energy': 0.0035307638946926014, 'residual_energy': 0.000786945830775643}, {'rows': 128, 'own_energy': 5.290580055517596e-07, 'delta_energy': 3.2241955528832706e-08, 'residual_energy': 3.0749281824939327}, {'rows': 128, 'own_energy': 0.008602503733035605, 'delta_energy': 0.002180600646633589, 'residual_energy': 0.0006913081628599597}, {'rows': 128, 'own_energy': 0.0008401817363806387, 'delta_energy': 2.4692279700181073e-05, 'residual_energy': 2.8428000909303404e-05}, {'rows': 128, 'own_energy': 0.0007571526717382862, 'delta_energy': 0.0003626325348499983, 'residual_energy': 0.00021435160216946432}, {'rows': 128, 'own_energy': 0.00047710960827313886, 'delta_energy': 0.00018749133328207124, 'residual_energy': 0.00011353137296112507}, {'rows': 128, 'own_energy': 0.04840046762552558, 'delta_energy': 0.03806134980237087, 'residual_energy': 0.015373173548091682}, {'rows': 87, 'own_energy': 0.43185587261628167, 'delta_energy': 0.27754343203422066, 'residual_energy': 0.1254564283724513}, {'rows': 128, 'own_energy': 0.000689201259075347, 'delta_energy': 0.00017058680757839925, 'residual_energy': 9.343927657117877e-05}, {'rows': 128, 'own_energy': 0.0008555377719667053, 'delta_energy': 1.1350902911181895e-05, 'residual_energy': 2.6027820377388323e-05}, {'rows': 128, 'own_energy': 1.879793952170209e-07, 'delta_energy': 1.4168337375527346e-07, 'residual_energy': 4.531008947988467e-05}, {'rows': 128, 'own_energy': 0.00011848235698991091, 'delta_energy': 1.810096339307701e-06, 'residual_energy': 3.540488876369788e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1978, 'test_rows': 1978}, 'scan_rows': 136358, 'valid_scan_rows': 136358, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 18, 'classes': [{'rows': 128, 'own_energy': 0.0027465052379791292, 'delta_energy': 0.0021440090909210134, 'residual_energy': 0.00038535867525042576}, {'rows': 76, 'own_energy': 0.021024029425872073, 'delta_energy': 0.007906046378147943, 'residual_energy': 0.0029600974001985584}, {'rows': 128, 'own_energy': 4.7604241870491124e-05, 'delta_energy': 3.098970110706185e-05, 'residual_energy': 9.919179882846107e-06}, {'rows': 77, 'own_energy': 0.01094303887104323, 'delta_energy': 0.007743233033412821, 'residual_energy': 0.002929882945994465}, {'rows': 128, 'own_energy': 0.0016002910107630517, 'delta_energy': 0.0001926220174686508, 'residual_energy': 3.7122076276165845e-05}, {'rows': 128, 'own_energy': 0.00011238326733837304, 'delta_energy': 8.063791031770286e-05, 'residual_energy': 6.865538810139758e-05}, {'rows': 35, 'own_energy': 0.07730666401108227, 'delta_energy': 0.04507834179652488, 'residual_energy': 0.020904913445690468}, {'rows': 47, 'own_energy': 0.01745700226255367, 'delta_energy': 0.01440418790575596, 'residual_energy': 0.003210800159975201}, {'rows': 128, 'own_energy': 0.00013917683106327882, 'delta_energy': 7.177506732439942e-06, 'residual_energy': 5.166193211813962e-06}, {'rows': 128, 'own_energy': 1.3641850670602102e-06, 'delta_energy': 1.0947219133483935e-05, 'residual_energy': 4.700130783333537e-06}, {'rows': 128, 'own_energy': 1.4584915776806253e-05, 'delta_energy': 0.00015510588665256734, 'residual_energy': 7.387006048112537e-05}, {'rows': 128, 'own_energy': 0.0012052766309799264, 'delta_energy': 0.01159756097175517, 'residual_energy': 0.001956178873819822}, {'rows': 128, 'own_energy': 0.0001079090230745607, 'delta_energy': 0.000313491899576837, 'residual_energy': 8.89493739108718e-05}, {'rows': 128, 'own_energy': 0.0069371142346593756, 'delta_energy': 0.003344279473859175, 'residual_energy': 0.0006365934110015596}, {'rows': 128, 'own_energy': 4.63307296982732e-06, 'delta_energy': 4.627061943243676e-09, 'residual_energy': 1.8296953348106383e-06}, {'rows': 128, 'own_energy': 0.00011303328321093749, 'delta_energy': 0.00023986101577953378, 'residual_energy': 5.910646644067512e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1770, 'test_rows': 1771}, 'scan_rows': 24804, 'valid_scan_rows': 24804, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 19, 'classes': [{'rows': 128, 'own_energy': 4.926618996178686e-05, 'delta_energy': 8.533383744213028e-06, 'residual_energy': 1.6506846673368545e-06}, {'rows': 128, 'own_energy': 0.0011020169030820338, 'delta_energy': 3.741391130691803e-05, 'residual_energy': 9.33242091625041e-06}, {'rows': 128, 'own_energy': 2.6364572098581176e-05, 'delta_energy': 2.013945718262116e-05, 'residual_energy': 6.950578931296105e-06}, {'rows': 128, 'own_energy': 0.0013562245915062052, 'delta_energy': 0.00011839900820924508, 'residual_energy': 3.147299623511644e-05}, {'rows': 67, 'own_energy': 0.023557078978486346, 'delta_energy': 0.018370995748550927, 'residual_energy': 0.0073776957610299215}, {'rows': 128, 'own_energy': 7.154555987415105e-08, 'delta_energy': 5.193019591729924e-08, 'residual_energy': 3.857335387486025e-06}, {'rows': 128, 'own_energy': 0.005707926522165591, 'delta_energy': 0.000164309569993447, 'residual_energy': 4.111552290665416e-05}, {'rows': 80, 'own_energy': 0.011074688419827833, 'delta_energy': 0.008136316636982669, 'residual_energy': 0.0024607491734398887}, {'rows': 32, 'own_energy': 0.06276982970449607, 'delta_energy': 0.05601289387740395, 'residual_energy': 0.008519811375191763}, {'rows': 128, 'own_energy': 6.778914341594862e-07, 'delta_energy': 6.821550860402509e-06, 'residual_energy': 4.926202499803834e-06}, {'rows': 128, 'own_energy': 0.0011008598398106711, 'delta_energy': 1.566261340738756e-05, 'residual_energy': 6.223701135765545e-06}, {'rows': 39, 'own_energy': 0.03269565756186062, 'delta_energy': 0.010310092965090716, 'residual_energy': 0.003839854911284828}, {'rows': 128, 'own_energy': 0.007163389603299463, 'delta_energy': 0.001721029048554268, 'residual_energy': 0.00037999639648149646}, {'rows': 128, 'own_energy': 0.003053481189358406, 'delta_energy': 0.0011113198199343597, 'residual_energy': 0.0001943138780973561}, {'rows': 128, 'own_energy': 5.585321340672219e-05, 'delta_energy': 3.1746811233860324e-05, 'residual_energy': 8.660934169496028e-06}, {'rows': 128, 'own_energy': 0.0006111380098278534, 'delta_energy': 3.324253986583908e-05, 'residual_energy': 1.0524016527524117e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1752, 'test_rows': 1754}, 'scan_rows': 27151, 'valid_scan_rows': 27151, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}]}

assert hashlib.sha256(DIAGNOSTIC.encode()).hexdigest()==DIAGNOSTIC_SHA
(WORK/'proj/rank_projection_diagnostic.py').write_text(DIAGNOSTIC)
(SOURCE/'rank_projection_diagnostic.py').write_text(DIAGNOSTIC)
(SOURCE/'diagnostic_manifest.json').write_text(json.dumps(dict(sha256=DIAGNOSTIC_SHA,parent=REPLAY['parent'],audit_sha=REPLAY['audit_sha'])))
(OUT/'replay.json').write_text(json.dumps(REPLAY))


In [ ]:
SEED=43
SESSION_HOURS=.25

wb=wandb.init(project='mobihet-cross-client-rank-probe',id=f'ccr_diag_s{SEED}-{DIAGNOSTIC_SHA[:8]}',
    config=dict(seed=SEED,diagnostic_sha=DIAGNOSTIC_SHA,parent=REPLAY['parent'],no_training=True),mode='online',
    settings=wandb.Settings(save_code=False,disable_code=True))
assert not wb.settings._offline
active=[]
for shard in (0,1):
    assert time.monotonic()-SESSION_T0<SESSION_HOURS*3600-120,'budget exhausted before diagnostic'
    log=open(OUT/f'diagnostic_gpu{shard}.log','w')
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(shard),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
    p=subprocess.Popen([sys.executable,str(WORK/'proj/rank_projection_diagnostic.py'),'--input',str(SRC),
        '--out',str(OUT),'--seed',str(SEED),'--shard',str(shard)],cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
    active.append((p,log))
while active:
    assert time.monotonic()-SESSION_T0<SESSION_HOURS*3600,'diagnostic budget exhausted'
    for p,log in list(active):
        if p.poll() is None:continue
        log.close();active.remove((p,log));assert p.returncode==0,('diagnostic worker failed',p.returncode)
    if active:time.sleep(2)
rows=[json.loads((OUT/f'diagnostic_{k:02d}.json').read_text()) for k in range(20)]
assert all(r['original_energies_reproduced'] for r in rows)
result=dict(seed=SEED,parent=REPLAY['parent'],all_original_energies_reproduced=True,models=20,
    diagnostic_sha=DIAGNOSTIC_SHA,model_training=False,holdout_read=False,gate_changed=False,
    elapsed_seconds=time.monotonic()-SESSION_T0)
(OUT/'diagnostic_result.json').write_text(json.dumps(result,indent=2))
wb.log(result);wb.finish()
print('DIAGNOSTIC COMPLETE',result,flush=True)
